# Find survey intersections

In [16]:
%load_ext autoreload
%autoreload 2


import logging

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.io as pio

import airbornegeo

# setup logging to get some additional info from the airbornegeo functions
logging.getLogger("airbornegeo").setLevel("INFO")
logging.basicConfig()
pio.renderers.default = "notebook"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

This is a subset of the BAS AGAP survey over Antarctica's Gamburtsev Subglacial Mountains. The file is download and subset in the notebook `AGAP_magnetic_survey`, and the BAS processing steps are repeated in the notebook `processing_AGAP_magnetic_survey`.

In [31]:
data_df = pd.read_csv("data/AGAP_magnetic_survey_processed.csv")
data_df = data_df[
    [
        "easting",
        "northing",
        "height",
        "line",
        "unixtime",
        "distance_along_line",
        "mag",
    ]
]

data_df = data_df.rename(columns={"distance_along_line": "along_line_distance"})

# for testing limit number of lines
data_df = data_df[data_df.line.isin([149, 162, 47, 64, 80])]
data_df = data_df[
    ~(data_df.line == 47) | ~(data_df.along_line_distance.between(125e3, 134e3))
]
data_df = data_df[~(data_df.line == 64) | ~(data_df.along_line_distance > 497e3)]

# define line types: 0 for survey lines, 1 for tie lines
data_df["line_type"] = np.where(data_df.line >= 142, 1, 0)

data_df.head()

,easting,northing,height,line,unixtime,along_line_distance,mag,line_type
243447,1.078385e+06,324738.995745,4188.1,47,1.230251e+09,0.000000,-60.95,0
243448,1.078375e+06,324738.036077,4188.1,47,1.230251e+09,10.049631,-62.39,0
243449,1.078331e+06,324733.833216,4188.3,47,1.230251e+09,53.589032,-63.66,0
243450,1.078265e+06,324727.295388,4188.6,47,1.230251e+09,120.672054,-64.86,0
243451,1.078198e+06,324720.699627,4188.8,47,1.230251e+09,187.743476,-66.00,0


In [32]:
data_df[data_df.line_type == 0].line.unique()

array([47, 64, 80])

In [33]:
data_df[data_df.line_type == 1].line.unique()

array([149, 162])

In [34]:
airbornegeo.plotly_points(
    data_df,
    color_col="line",
    hover_cols=["along_line_distance", "line_type"],
)

## Calculate intersections of lines and ties

In [35]:
# convert dataframe into geodataframe
data_df = gpd.GeoDataFrame(
    data_df,
    geometry=gpd.points_from_xy(data_df.easting, data_df.northing),
    crs="EPSG:3031",
)

In [36]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
    line_column="line",
    method="pairs",
)
inters

Line pairs:   0%|          | 0/6 [00:00<?, ?it/s]

INFO:airbornegeo:found 5 intersections


,line1,line2,geometry,max_dist,easting,northing
0,47,149,POINT (590788 237703),12.785759,590788.0,237703.0
1,47,162,POINT (947875 301515),1470.097394,947875.0,301515.0
2,64,162,POINT (961606 224280),13.818089,961606.0,224280.0
3,80,149,POINT (616278 93894),8.205492,616278.0,93894.0
4,80,162,POINT (973552 156679),22.552582,973552.0,156679.0


In [37]:
airbornegeo.plotly_points(
    inters,
    color_col="max_dist",
    hover_cols=["line1", "line2"],
    robust=True,
    size=30,
    edge_width=1,
    theme=None,
    cmap="matter",
    title="Distance from intersection to nearest data point",
)

From the above map and table we can see there are 5 intersection points. You can hover your mouse over the points in the plot to see some more information about them, including which line and tie they belong to, and the distance from the intersection point to the nearest actual measurement of the line and tie. 

Theoretically since there are 2 ties and 3 lines, there could be 6 intersection points. Notice that line 64 ends just before crossing tie 149. If you zoom in on the flight survey plot, you can see this. 

### Buffer the ends of each line
Since those two lines would cross if line 64 was just a few hundred meters longer, we can buffer all the lines to see if you can include it as an intersection. We show this below with the `buffer_dist` parameter.

In [38]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
    line_column="line",
    method="pairs",
    buffer_dist=200,  # if intersection would be within 200 m of end of line, include it
)
inters

Line pairs:   0%|          | 0/6 [00:00<?, ?it/s]

INFO:airbornegeo:found 6 intersections


,line1,line2,geometry,max_dist,easting,northing
0,47,149,POINT (590788 237703),12.785759,590788.0,237703.0
1,47,162,POINT (947753 301493),1346.129682,947753.0,301493.0
2,64,149,POINT (604391 161048),173.862216,604391.0,161048.0
3,64,162,POINT (961606 224280),13.818089,961606.0,224280.0
4,80,149,POINT (616278 93894),8.205492,616278.0,93894.0
5,80,162,POINT (973552 156679),22.552582,973552.0,156679.0


In [39]:
airbornegeo.plotly_points(
    inters,
    color_col="max_dist",
    hover_cols=["line1", "line2"],
    robust=True,
    size=30,
    edge_width=1,
    theme=None,
    cmap="matter",
    title="Distance from intersection to nearest data point",
)

Now you can see we've included this 6th intersection, and it was only ~170m away from the end of line 64. Notice that the intersection of lines 47 and 162 is over 1 km away from the nearest actual data point of line 47. This might be too far away to be useful as an intersection. 

### Exclude intersections far away from data
Below we show how the `cutoff_dist` parameter can be used to exclude intersections which are far away from either line's data. Each intersection point has a `max_dist` parameter, which is the largest of the distances between the point and the nearest data point of either line.

In [40]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
    line_column="line",
    method="pairs",
    buffer_dist=500,  # if intersection would be within 500 m of end of line, include it
    cutoff_dist=1e3,  # if intersection is more than 1km from nearest data point, it is not included
)
inters

Line pairs:   0%|          | 0/6 [00:00<?, ?it/s]

INFO:airbornegeo:found 6 intersections
INFO:airbornegeo:removed 1 intersection point(s) with a max distance greater than 1 km


,line1,line2,geometry,max_dist,easting,northing
0,47,149,POINT (590788 237703),12.785759,590788.0,237703.0
1,64,149,POINT (604391 161048),173.862216,604391.0,161048.0
2,64,162,POINT (961606 224280),13.818089,961606.0,224280.0
3,80,149,POINT (616278 93894),8.205492,616278.0,93894.0
4,80,162,POINT (973552 156679),22.552582,973552.0,156679.0


In [41]:
airbornegeo.plotly_points(
    inters,
    color_col="max_dist",
    hover_cols=["line1", "line2"],
    robust=True,
    size=30,
    edge_width=1,
    theme=None,
    cmap="matter",
    title="Distance from intersection to nearest data point",
)

We have now successfull included the intersection which was within 200 m of the end of a line using the `buffer_dist`, and also excluded the intersection which was over 1 km away from the nearest data point, using `cutoff_dist`.

### Manually exclude intersections

We can manually exclude intersections if we know they have issues. You can supply tuples of pairs on line and tie numbers, or if you want to exclude an entire line or tie, just supply the individual line or tie number.

In [42]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
    line_column="line",
    method="pairs",
    buffer_dist=500,  # if intersection would be within 500 m of end of line, include it
    cutoff_dist=1e3,  # if intersection is more than 1km from nearest data, it is not included
    exclude_ints=[
        [47],
        [80, 149],
    ],  # exclude all intersections of line 47 and the intersection of 80 and 149
)
inters

Line pairs:   0%|          | 0/6 [00:00<?, ?it/s]

INFO:airbornegeo:found 6 intersections
INFO:airbornegeo:removed 1 intersection point(s) with a max distance greater than 1 km
INFO:airbornegeo:manually omitted 2 intersections points


,line1,line2,geometry,max_dist,easting,northing
0,64,149,POINT (604391 161048),173.862216,604391.0,161048.0
1,64,162,POINT (961606 224280),13.818089,961606.0,224280.0
2,80,162,POINT (973552 156679),22.552582,973552.0,156679.0


In [43]:
airbornegeo.plotly_points(
    inters,
    color_col="max_dist",
    hover_cols=["line1", "line2"],
    robust=True,
    size=30,
    edge_width=1,
    theme=None,
    cmap="matter",
    title="Distance from intersection to nearest data point",
)

## Network intersections
In the above examples, we set the `method` parameter to 'pairs'. This find only the intersections between the two category of line types, (survey lines and orthogonal tie lines), which are defined in the column `line_type` by values of 0 and 1. Below we show how to calculate all possible intersections between any lines, which may include intersections between two lines of the same type.

In [44]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
    line_column="line",
    method="network",
)
inters

Line pairs:   0%|          | 0/10 [00:00<?, ?it/s]

INFO:airbornegeo:found 5 intersections


,line1,line2,geometry,max_dist,easting,northing
0,47,149,POINT (590788 237703),12.785759,590788.0,237703.0
1,47,162,POINT (947875 301515),1470.097394,947875.0,301515.0
2,64,162,POINT (961606 224280),13.818089,961606.0,224280.0
3,80,149,POINT (616278 93894),8.205492,616278.0,93894.0
4,80,162,POINT (973552 156679),22.552582,973552.0,156679.0
